In [1]:
!pip install -U ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.9 MB/s eta 0:00:00


In [2]:
import ultralytics
from ultralytics import YOLO
import torch

ultralytics.checks()  # prints a full environment summary: python, torch, CUDA, GPU

print("\nCUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.6/8062.4 GB disk)

CUDA available: True
Device: Tesla T4


In [3]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [4]:
import yaml

data_yaml = {
    'path': '/kaggle/input/datasets/kefanzhan/diordataset/Dior/Dior',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'airplane', 1: 'airport', 2: 'baseballfield', 3: 'basketballcourt', 4: 'bridge',
        5: 'chimney', 6: 'dam', 7: 'expresswayservicearea', 8: 'expresswaytollstation',
        9: 'golffield', 10: 'groundtrackfield', 11: 'harbor', 12: 'overpass', 13: 'ship',
        14: 'stadium', 15: 'storagetank', 16: 'tenniscourt', 17: 'trainstation',
        18: 'vehicle', 19: 'windmill'
    }
}

yaml_path = '/kaggle/working/dior.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(open(yaml_path).read())

path: /kaggle/input/datasets/kefanzhan/diordataset/Dior/Dior
train: images/train
val: images/val
names:
  0: airplane
  1: airport
  2: baseballfield
  3: basketballcourt
  4: bridge
  5: chimney
  6: dam
  7: expresswayservicearea
  8: expresswaytollstation
  9: golffield
  10: groundtrackfield
  11: harbor
  12: overpass
  13: ship
  14: stadium
  15: storagetank
  16: tenniscourt
  17: trainstation
  18: vehicle
  19: windmill



In [5]:
import torch
import torch._utils  # explicitly import so torch's lazy attribute loader doesn't fail

from ultralytics import YOLO
model = YOLO("yolo26n.pt")
print("Model loaded fine:", model)

Model loaded fine: YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=T

In [6]:
from ultralytics import YOLO
import shutil, os

def backup_checkpoint(trainer):
    src = trainer.save_dir / 'weights' / 'last.pt'
    if src.exists():
        os.makedirs('/kaggle/working/checkpoint_backups', exist_ok=True)
        shutil.copy(src, '/kaggle/working/checkpoint_backups/last_backup.pt')

model_dior = YOLO("yolo26n.pt")
model_dior.add_callback("on_train_epoch_end", backup_checkpoint)

model_dior.train(
    data='/kaggle/working/dior.yaml',
    epochs=50,
    imgsz=800,
    batch=16,
    project='/kaggle/working/runs',
    name='dior_full_v2',
    seed=42,
    deterministic=True,
    exist_ok=True,
    patience=15,
    cache=True,
    save_period=5,
)

Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dior.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dior_full_v2, nbs=64, nms=None, opset=None, optimize=

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      8.97G      1.323       2.89   0.006994        159        800: 100% ━━━━━━━━━━━━ 733/733 3.7it/s 3:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 367/367 2.6it/s 2:22
                   all      11738     124443      0.659      0.483      0.506      0.329

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/50      6.86G      1.329      1.922   0.007033        116        800: 100% ━━━━━━━━━━━━ 733/733 3.6it/s 3:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 367/367 2.6it/s 2:21
                   all      11738     124443      0.701      0.532      0.567      0.371

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/50      8.03G      1.317      1.538   0.006911        199        800: 100% ━━━━━━━━━━━━ 733/733 3.7it/s 3:21
                 Class     Images  Instances      Bo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ab65c5fa720>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043